# Dashboards con Grafana

Visualizar métricas y construir dashboards de monitoreo

## Introducción

Un buen dashboard operativo responde si el servicio está sano y dónde investigar una degradación. No es un panel de vanity metrics — debe facilitar decisiones rápidas durante incidentes.

### Objetivos de Aprendizaje

- Entender qué debe responder un buen dashboard operativo
- Conocer los tipos de paneles y visualizaciones
- Construir dashboards que faciliten debugging
- Aplicar alertas basadas en umbrales
- Diseñar dashboards para diferentes audiencias

## Qué debe responder un buen dashboard operativo

> Un buen dashboard responde: ¿está sano el servicio? Si no, ¿dónde está el problema y qué métrica investigar? Debe permitir toma de decisiones en segundos, no investigación profunda.

In [ ]:
print("=== Propósito de un Dashboard Operativo ===")

preguntas_dashboard = {
    "¿Está sano?": "Estado general del servicio (OK/Degradado/Down)",
    "¿Qué servicio está mal?": "Segmentación por componente",
    "¿Desde cuándo?": "Tendencia temporal",
    "¿Qué métrica investigar?": "Drill-down a la métrica específica",
    "¿Está afectando usuarios?": "Error rate, latencia用户体验",
}

print("Un dashboard debe responder:\n")
for pregunta, desc in preguntas_dashboard.items():
    print(f"  {pregunta}: {desc}")

print("""
Principios de un buen dashboard:
  1. Información arriba-abajo: lo más crítico primero
  2. Tiempo a la izquierda: eventos recientes
  3. Acciones posibles: cada panel debe sugerir qué hacer
  4. Tolerante a fallos: si una fuente falla, el resto funciona
""")

## Tipos de Paneles y Visualizaciones

> Grafana soporta múltiples tipos de visualización. Escoger el correcto según el tipo de dato: series temporales (time series), valores únicos (stat), estados (gauge), distribución (histogram).

In [ ]:
print("=== Tipos de Paneles ===")

tipos_paneles = {
    "Time Series": "Series temporales, tendencias. Para rates y contadores.",
    "Stat": "Valor único grande. Para métricas instantáneas.",
    "Gauge": "Medidor con umbrales. Para uso de recursos.",
    "Bar Gauge": "Barras horizontales. Para comparar valores.",
    "Table": "Datos tabulares. Para listas de instancias.",
    "Pie Chart": "Proporciones. Para distribución.",
    "Heatmap": "Densidad. Para distribución de latencia.",
    "Logs": "Logs estructurados. Para debugging.",
}

for tipo, uso in tipos_paneles.items():
    print(f"  {tipo}: {uso}")

print("""
Ejemplos de uso:
  Time Series: Tasa de requests por segundo
  Stat: Requests totales hoy
  Gauge: Uso de CPU (0-100%)
  Table: Instancias con mayor latencia
""")

## Diseño de Layout para Incidentes

> Un dashboard para incidentes debe tener información crítica arriba: estado general, errores, latencia. Debajo, detalles para drill-down.

In [ ]:
print("=== Layout para Incidentes ===")

layout = """
┌─────────────────────────────────────────────────────┐
│  ESTADO GENERAL: ● OK / ● DEGRADADO / ● DOWN      │
├─────────────────────────────────────────────────────┤
│  [Requests/s] [Error Rate %] [Latencia p99]       │
│       stat          stat           stat             │
├─────────────────────────────────────────────────────┤
│  Requests por segundo (time series)                │
│  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~          │
├─────────────────────────────────────────────────────┤
│  Error rate por endpoint (time series)             │
│  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~          │
├─────────────────────────────────────────────────────┤
│  Latencia p50/p95/p99 (time series)                │
│  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~          │
├─────────────────────────────────────────────────────┤
│  Detalle por instancia (table)                     │
│  Instancia | CPU | Memory | Latency                │
└─────────────────────────────────────────────────────┘
"""

print(layout)

print("Regla: información que responde '¿está mal?' debe estar visible sin scroll.")

## Variables y Filtros Dinámicos

> Las variables de Grafana permiten crear dashboards interactivos. Una variable como $service puede filtrar todos los paneles simultáneamente.

In [ ]:
print("=== Variables de Dashboard ===")

variables = {
    "service": "auth, api, billing, inventory (query: label_values(http_requests_total, service))",
    "environment": "production, staging, development",
    "region": "us-east, us-west, eu-central",
    "instance": "instance-01, instance-02 (query: label_values(node_cpu_seconds_total, instance))",
}

print("Variables comunes:\n")
for nombre, desc in variables.items():
    print(f"  ${nombre}: {desc}")

print("\nUso en queries:")
print('  sum(rate(http_requests_total{service="$service"}[1m]))')
print('  node_memory_MemAvailable{instance=~"$instance.*"}')

print("\nEl usuario selecciona el valor y todos los paneles se actualizan.")

## Alertas en Grafana

> Las alertas de Grafana se evalúan continuamente y notifican cuando se violan umbrales. Configura canales de notificación (Slack, PagerDuty, email) y reglas de alert.

In [ ]:
print("=== Alertas en Grafana ===")

alerta_ejemplo = """
Alert: HighErrorRate
Condition: avg(http_requests_total{status=~"5.."}) / avg(http_requests_total) * 100 > 5
For: 5m
Severity: critical

Notification:
  - Slack: #alerts-prod
  - PagerDuty: on-call-engineer
"""

print("Ejemplo de regla de alerta:\n")
print(alerta_ejemplo)

print("Configurar alertas:")
pasos_alerta = [
    "1. Crear panel con la métrica a alertar",
    "2. Click en título > Edit > Alerts",
    "3. Create alert rule",
    "4. Definir condición: avg() > threshold",
    "5. Configurar for: 5m (espera antes de notificar)",
    "6. Asignar notification channel",
]
for paso in pasos_alerta:
    print(f"  {paso}")

print("\nfor: 5m significa que la condición debe mantenerse 5 min antes de disparar.")
print("Esto evita alertas por spikes transitorios.")

## Datasources y Provisioning

> Grafana soporta múltiples datasources: Prometheus, Elasticsearch, InfluxDB, PostgreSQL. Para dashboards reproducible, usa provisioning con archivos YAML.

In [ ]:
print("=== Datasources ===")

datasources = {
    "Prometheus": "Métricas de aplicaciones y sistema",
    "Elasticsearch": "Logs estructurados",
    "InfluxDB": "Series temporales IoT/sensores",
    "PostgreSQL": "Métricas de base de datos",
    "CloudWatch": "AWS metrics y logs",
}

print("Datasources comunes:\n")
for ds, desc in datasources.items():
    print(f"  {ds}: {desc}")

provisioning = """
# /etc/grafana/provisioning/datasources/prometheus.yml
apiVersion: 1

datasources:
  - name: Prometheus
    type: prometheus
    access: proxy
    url: http://prometheus:9090
    isDefault: true
"""

print("\nProvisioning de datasource:")
print(provisioning)

## Dashboards por Caso de Uso

> Diferentes audiencias necesitan diferentes dashboards: equipo de desarrollo para debugging, SRE para SLAs, negocio para métricas de producto.

In [ ]:
print("=== Dashboards por Audiencia ===")

dashboards = {
    "Desarrollo": {
        "Proposito": "Debugging y desarrollo",
        "Paneles": "Logs detallados, traces, errores por componente",
    },
    "SRE/DevOps": {
        "Proposito": "Disponibilidad y SLAs",
        "Paneles": "SLOs, error budget, uptime, capacidad",
    },
    "Negocio": {
        "Proposito": "Métricas de producto",
        "Paneles": "Usuarios activos, conversión, revenue",
    },
    "Incidente": {
        "Proposito": "Responder a outages",
        "Paneles": "Estado general, errores, latencia, qué investigar",
    },
}

for audiencia, info in dashboards.items():
    print(f"\n{audiencia}:")
    for k, v in info.items():
        print(f"  {k}: {v}")

## Tips y Mejores Prácticas

> Un dashboard debe responder en 5 segundos: ¿está bien o está mal? Si necesita scroll para verlo, tiene demasiados paneles.

> Usa colores consistentes: verde=OK, amarillo=Warning, rojo=Critical. No inventes convenciones nuevas.

> El tiempo default del dashboard debe ser "last 15 minutes" — el caso de uso principal es ver qué está pasando ahora.

> Limita el número de métricas visibles: 6-8 paneles es el máximo antes de que la información se vuelva ruido.

> Haz que cada panel sea independently meaningful: alguien debería poder entender un panel sin contexto de los demás.

## Errores Comunes

### Dashboard informativo pero no accionable

¿Por qué ocurre?
- Se muestran demasiadas métricas sin priorizar.

Solución
- Antes de añadir un panel, pregúntate: ¿qué haré si esto cambia? Si no hay respuesta, no es necesario.

### No probar las alertas

¿Por qué ocurre?
- Se configura la alerta y se olvida hasta que dispara en un incidente real.

Solución
- Prueba las alertas periódicamente. Usa "Test" en Grafana para verificar que la condición dispara.

### Promedio en vez de percentiles

¿Por qué ocurre?
- Se monitorea avg(latency) cuando el p99 es lo que用户体验.

Solución
- Para latencia, muestra p50, p95, p99. El promedio oculta outliers que son exactamente lo que importa.

### Dashboards con demasiados paneles

¿Por qué ocurre?
- Se añaden paneles "por si acaso" sin criterio.

Solución
- Máximo 6-8 paneles. Si necesitas más, crea múltiples dashboards temáticos.

### No considerar el costo de queries

¿Por qué ocurre?
- Queries muy frecuentes o con joins complejos saturando Prometheus.

Solución
- Revisa el costo de las queries. Grafana muestra el tiempo de ejecución en inspect > Stats.